# 05 — Error & Attention Analysis

Analyze translation errors, visualize attention heatmaps, and identify
systematic failure patterns.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import pandas as pd

from nmt.evaluation.error_analysis import ErrorAnalyzer
from nmt.evaluation.metrics import compute_bleu
from nmt.utils.logging import setup_logging

setup_logging('INFO')
sns.set_theme(style='white')

## 1. Load Predictions

In [ ]:
# Load hypotheses / references from prediction files
# Example:
# with open('../artifacts/predictions/amharic/attention_seq2seq_predictions.txt') as f:
#     hypotheses = [l.strip() for l in f]
# with open('../data/processed/amharic/test.am') as f:
#     references = [l.strip() for l in f]
# with open('../data/processed/amharic/test.en') as f:
#     src_sentences = [l.strip() for l in f]

# Placeholders
src_sentences, hypotheses, references = [], [], []

## 2. Overall BLEU & chrF

In [ ]:
if hypotheses:
    metrics = compute_bleu(hypotheses, references)
    for k, v in metrics.items():
        print(f'{k:12s}: {v:.4f}')

## 3. Error Analysis

In [ ]:
if hypotheses:
    analyzer = ErrorAnalyzer(src_sentences, hypotheses, references)

    print('--- Length Ratio Stats ---')
    for k, v in analyzer.length_ratio_stats().items():
        print(f'  {k}: {v:.3f}')

    print('\n--- Sentence BLEU Distribution ---')
    dist = analyzer.sentence_bleu_distribution()
    for bucket, count in dist.items():
        print(f'  BLEU {bucket}: {count}')

    print('\n--- Most Missed Target Tokens (top 20) ---')
    for token, count in analyzer.most_common_errors(top_k=20):
        print(f'  {token!r}: {count}')

## 4. Worst Translations

In [ ]:
if hypotheses:
    worst = analyzer.get_worst_translations(n=5)
    for i, ex in enumerate(worst, 1):
        print(f'[{i}] BLEU={ex["bleu"]:.2f}')
        print(f'  SRC : {ex["src"]}')
        print(f'  HYP : {ex["hypothesis"]}')
        print(f'  REF : {ex["reference"]}')
        print()

## 5. Attention Heatmap

In [ ]:
def plot_attention(attention, src_tokens, tgt_tokens, title='Attention', save_path=None):
    """Plot an attention weight heatmap.
    
    Args:
        attention: 2D array of shape (tgt_len, src_len).
        src_tokens: List of source token strings.
        tgt_tokens: List of target token strings.
        title: Plot title.
        save_path: Optional path to save the figure.
    """
    attention = np.array(attention)
    fig, ax = plt.subplots(figsize=(max(6, len(src_tokens) * 0.6), max(4, len(tgt_tokens) * 0.5)))
    im = ax.imshow(attention, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(tgt_tokens)))
    ax.set_yticklabels(tgt_tokens, fontsize=9)

    ax.set_xlabel('Source tokens')
    ax.set_ylabel('Target tokens')
    ax.set_title(title)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


# Example usage with dummy data:
# attention_weights = ...  # shape (tgt_len, src_len) tensor or array
# plot_attention(
#     attention_weights.cpu().numpy(),
#     src_tokens=['The', 'cat', 'sat', '<eos>'],
#     tgt_tokens=['ድመቷ', 'ተቀምጧል', '<eos>'],
#     save_path='../reports/figures/attention_example.png'
# )